## 1. Carregamento dos dados + Definição do problema

In [26]:
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("modified_data.csv")
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,price_per_sqft
0,2014-05-02,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,233.58
1,2014-05-02,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,653.15
2,2014-05-02,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,177.20
3,2014-05-02,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,210.00
4,2014-05-02,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,283.51


**Problema:** regressão — prever `price` (preço de venda) a partir das
características do imóvel e sua localização.

**Alvo:** `price`.

## 2. Exploração básica

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            4600 non-null   object 
 1   price           4600 non-null   float64
 2   bedrooms        4600 non-null   float64
 3   bathrooms       4600 non-null   float64
 4   sqft_living     4600 non-null   int64  
 5   sqft_lot        4600 non-null   int64  
 6   floors          4600 non-null   float64
 7   waterfront      4600 non-null   int64  
 8   view            4600 non-null   int64  
 9   condition       4600 non-null   int64  
 10  sqft_above      4600 non-null   int64  
 11  sqft_basement   4600 non-null   int64  
 12  yr_built        4600 non-null   int64  
 13  yr_renovated    4600 non-null   int64  
 14  street          4600 non-null   object 
 15  city            4600 non-null   object 
 16  statezip        4600 non-null   object 
 17  price_per_sqft  4600 non-null   f

In [4]:
# Verify if there are any missing values in the dataset
df.isnull().sum().sort_values(ascending=False)

date              0
price             0
bedrooms          0
bathrooms         0
sqft_living       0
sqft_lot          0
floors            0
waterfront        0
view              0
condition         0
sqft_above        0
sqft_basement     0
yr_built          0
yr_renovated      0
street            0
city              0
statezip          0
price_per_sqft    0
dtype: int64

Parece que não há nulos, mas temos que checar os valores que podem esconder isso.

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
price,4600.0,551962.988470,563834.702545,0.0,322875.0000,460943.460,654962.50,26590000.0
bedrooms,4600.0,3.400870,0.908848,0.0,3.0000,3.000,4.00,9.0
bathrooms,4600.0,2.160815,0.783781,0.0,1.7500,2.250,2.50,8.0
sqft_living,4600.0,2139.346957,963.206916,370.0,1460.0000,1980.000,2620.00,13540.0
sqft_lot,4600.0,14852.516087,35884.436145,638.0,5000.7500,7683.000,11001.25,1074218.0
floors,4600.0,1.512065,0.538288,1.0,1.0000,1.500,2.00,3.5
waterfront,4600.0,0.007174,0.084404,0.0,0.0000,0.000,0.00,1.0
view,4600.0,0.240652,0.778405,0.0,0.0000,0.000,0.00,4.0
condition,4600.0,3.451739,0.677230,1.0,3.0000,3.000,4.00,5.0
sqft_above,4600.0,1827.265435,862.168977,370.0,1190.0000,1590.000,2300.00,9410.0


Podemos observar:
- Prices 0 - mas como price será o alvo, não vamos imputar. Vamos descartar essas linhas.

- price_per_sqft é vazamento de dados - (price / sqft_living), manter isso faria o modelo inferir diretamente. Prova de correlação na célula abaixo (0.82 alto demais).

- sqft_above reduntante (`sqft_above + sqft_basement = sqft_living` em 100% das linhas) - vamos descartar sqft_above para evitar carregar a redundância adiante.

- cardinalidade em colunas categóricas: 
      - `street`: praticamente um valor por linha (4525 de 4600) — one-hot é
  inviável (explodiria a dimensionalidade) e o valor bruto do endereço não
  generaliza. **Descartar.**
      - `statezip`: mistura estado (`WA`, constante em todo o dataset — zero
  informação) com CEP. Vamos extrair só o CEP em uma coluna nova e descartar o
  restante.
      - `city`: 44 categorias, cardinalidade moderada — candidata direta a
  `OneHotEncoder`.

- `yr_renovated == 0` significa "nunca reformado", não "ano zero".** - criar uma feature binária `was renovated` e manter `yr_renovated` apenas como contexto adicional.

- `date` cobre uma janela curta (maio–julho de 2014), então não
extrairemos ano de venda (constante); só o mês, caso haja sazonalidade.

- `bedrooms == 0` e/ou `bathrooms == 0`  são fisicamente implausíveis. São apenas 2 imóveis nessas condições, e isso indica um erro de coleta/dado ausente. Como são features (não o alvo) e representam apenas ~0.04% das linhas, a ação escolhida é descartar as duas linhas, em vez de imputar.

In [10]:
df[["price", "price_per_sqft"]].corr()

,price,price_per_sqft
price,1.000000,0.819279
price_per_sqft,0.819279,1.000000


In [11]:
(df["sqft_above"] + df["sqft_basement"] == df["sqft_living"]).mean()

np.float64(1.0)

In [12]:
for col in ["street", "city", "statezip"]:
    print(f"{col}: {df[col].nunique()} categorias únicas (de {len(df)} linhas)")

street: 4525 categorias únicas (de 4600 linhas)
city: 44 categorias únicas (de 4600 linhas)
statezip: 77 categorias únicas (de 4600 linhas)


In [14]:
print("Intervalo de datas:", df["date"].min(), "a", df["date"].max())

Intervalo de datas: 2014-05-02 a 2014-07-10


In [22]:
zero_rooms = df[(df["bedrooms"] == 0) | (df["bathrooms"] == 0)]
print(f"Linhas com bedrooms==0 ou bathrooms==0: {len(zero_rooms)}")
zero_rooms[["price", "bedrooms", "bathrooms", "sqft_living", "condition", "city"]]

Linhas com bedrooms==0 ou bathrooms==0: 2


,price,bedrooms,bathrooms,sqft_living,condition,city
2365,1095000.0,0.0,0.0,3064,3,Seattle
3209,1295648.0,0.0,0.0,4810,3,Redmond


## 3. Limpeza e escolha de features

In [ ]:
data = df.copy()

# 1: price == 0 é ausência disfarçada no alvo -> descarta a linha
data = data[data["price"] > 0].reset_index(drop=True)

# 2: price_per_sqft vaza o alvo -> descarta
data = data.drop(columns=["price_per_sqft"])

# 3: sqft_above é redundante (sqft_above + sqft_basement = sqft_living)
data = data.drop(columns=["sqft_above"])

# 4: street tem cardinalidade inviável -> descarta
#           statezip -> extrai só o CEP
data["zipcode"] = data["statezip"].str.extract(r"(\d{5})")
data = data.drop(columns=["street", "statezip"])

# 5: yr_renovated == 0 -> cria binária was_renovated
data["was_renovated"] = (data["yr_renovated"] != 0).astype(int)

# 6: extrai mês da venda (ano é constante nesse recorte)
data["sale_month"] = pd.to_datetime(data["date"]).dt.month
data = data.drop(columns=["date"])

# 7: bedrooms == 0 e bathrooms == 0 em imóveis de 3000-4800 sqft vendidos por mais de $1M -> fisicamente implausível.
# Apenas 2 linhas (~0.04%) -> descarta em vez de imputar.
n_before = len(data)
data = data[~((data["bedrooms"] == 0) & (data["bathrooms"] == 0))].reset_index(drop=True)
print(f"Linhas removidas por bedrooms==0 e bathrooms==0: {n_before - len(data)}")

print(data.shape)
data.describe().T

Linhas removidas por bedrooms==0 e bathrooms==0: 2
(4549, 16)


,count,mean,std,min,25%,50%,75%,max
price,4549.0,557625.653761,563891.453609,7800.00,326100.00,465000.00,657500.0,26590000.0
bedrooms,4549.0,3.396131,0.901988,1.00,3.00,3.00,4.0,9.0
bathrooms,4549.0,2.155968,0.775205,0.75,1.75,2.25,2.5,8.0
sqft_living,4549.0,2131.578809,955.234941,370.00,1460.00,1970.00,2610.0,13540.0
sqft_lot,4549.0,14834.598813,35971.144296,638.00,5000.00,7680.00,10970.0,1074218.0
floors,4549.0,1.511651,0.537794,1.00,1.00,1.50,2.0,3.5
waterfront,4549.0,0.006595,0.080949,0.00,0.00,0.00,0.0,1.0
view,4549.0,0.234337,0.765085,0.00,0.00,0.00,0.0,4.0
condition,4549.0,3.449549,0.675242,1.00,3.00,3.00,4.0,5.0
sqft_basement,4549.0,310.286876,462.043404,0.00,0.00,0.00,600.0,4820.0


## 4. Separação de features/alvo e `train_test_split`

O split acontece **antes** de qualquer transformador ser ajustado (`fit`), para
que o conjunto de teste simule dados nunca vistos.

In [27]:
X = data.drop(columns=["price"])
y = data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Treino:", X_train.shape, " Teste:", X_test.shape)

Treino: (3639, 15)  Teste: (910, 15)


## 5. Pré-processamento — Imputer, Scaler, Encoder via `ColumnTransformer`

Separação de colunas por tipo de tratamento:

| Grupo | Colunas | Imputer | Transformação |
|---|---|---|---|
| Numéricas contínuas/discretas | `bedrooms`, `bathrooms`, `sqft_living`, `sqft_lot`, `floors`, `sqft_basement`, `yr_built`, `yr_renovated`, `sale_month` | mediana | `StandardScaler` |
| Numéricas já ordinais/binárias | `waterfront`, `view`, `condition`, `was_renovated` | mediana | mantidas na escala original (ordinais de poucos níveis — escalonar não muda a informação para árvores, e para lineares o impacto é pequeno) |
| Categórica nominal | `city`, `zipcode` | mais frequente | `OneHotEncoder(handle_unknown="ignore")` |

O `SimpleImputer` é incluído em todos os grupos mesmo não havendo nulos
**neste** conjunto — é defensivo: garante que o pipeline não quebre se um novo
lote de dados chegar com valores ausentes.

In [29]:
numeric_scaled_cols = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "sqft_basement", "yr_built", "yr_renovated", "sale_month",
]
numeric_passthrough_cols = ["waterfront", "view", "condition", "was_renovated"]
categorical_cols = ["city", "zipcode"]

numeric_scaled_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
# por default, SimpleImputer(strategy="median") substitui valores ausentes (SimpleImputer(missing_values=np.nan, strategy="median")), e StandardScaler() padroniza os dados numéricos.

numeric_passthrough_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num_scaled", numeric_scaled_pipeline, numeric_scaled_cols),
    ("num_passthrough", numeric_passthrough_pipeline, numeric_passthrough_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

preprocessor

,transformers,"[('num_scaled', ...), ('num_passthrough', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None
